# Soundscape Study 2 — Circumplex Analysis
## ADHD · Autistic · Neurotypical  |  n = 30 per group

**Y-axis correction applied throughout:**  
Gorilla records Y increasing *downward* (top = 0, bottom = 1).  
The ISO circumplex places **Eventful at the top** (positive Y).  
Fix: `ISOEventful = 1 − Circumplex Y`  
ISOPleasant = Circumplex X (no change needed).

**Data files (30-30-30 balanced split):**
- `ADHD_Group_XY_n30.csv`
- `Autistic_Group_XY_n30.csv`
- `Neurotypical_Group_XY_n30.csv`
- `All_Groups_XY_n30-30-30.csv` (combined)

**Pipeline:**
1. Load & correct coordinates
2. Scatter + density circumplex plots (Soundscapy)
3. Per-soundscape faceted plots
4. Heatmaps (Gaussian KDE)
5. Multivariate Skew-Normal (MSN) fitting (Mitchell et al., 2024)
6. SPI via 2D Bivariate KS (Fasano & Franceschini, 1987)
7. Statistical tests + export

---
## Cell 1 — Environment Setup

In [ ]:
import sys, os
print(f"Python: {sys.version}")

# Load local circumplex-dev (Mitchell et al. dev branch)
circ_path = os.path.expanduser("~/Desktop/circumplex-dev")
sys.path.insert(0, circ_path)
print(f"circumplex-dev path exists: {os.path.exists(circ_path)}")

# Uncomment to install/upgrade if needed:
# !pip install --upgrade soundscapy[all]
# !pip install "git+https://github.com/MitchellAcoustics/circumplex@dev"

In [ ]:
# Cell 2 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import kstwobign, pearsonr, norm, multivariate_normal, skew
from scipy.optimize import differential_evolution
import warnings
warnings.filterwarnings('ignore')

# Patch rpy2 before soundscapy loads — neutralise deprecated activate() calls
try:
    import rpy2
    if not hasattr(rpy2, '__version__'):
        rpy2.__version__ = '0.0.0'
    import rpy2.robjects.numpy2ri as numpy2ri
    import rpy2.robjects.pandas2ri as pandas2ri
    numpy2ri.activate  = lambda: None   # no-op
    pandas2ri.activate = lambda: None   # no-op
except ImportError:
    pass

import soundscapy as sspy
from soundscapy.plotting import scatter, density
print(f"soundscapy: {sspy.__version__} ✓")

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

GROUP_COLOURS = {'ADHD': '#E69F00', 'Autistic': '#56B4E9', 'Neurotypical': '#009E73'}
print("All imports complete ✓")

---
## Cell 3 — 2D BKS Implementation (Fasano & Franceschini, 1987)

In [ ]:
def quadct(x, y, xx, yy):
    """Count fractions in each quadrant around point (x, y)."""
    n = len(xx)
    ix1, ix2 = xx <= x, yy <= y
    a = np.sum(ix1 & ix2) / n
    b = np.sum(ix1 & ~ix2) / n
    c = np.sum(~ix1 & ix2) / n
    d = 1 - a - b - c
    return a, b, c, d

def maxdist(x1, y1, x2, y2):
    n1 = len(x1)
    D1 = np.empty((n1, 4))
    for i in range(n1):
        a1, b1, c1, d1 = quadct(x1[i], y1[i], x1, y1)
        a2, b2, c2, d2 = quadct(x1[i], y1[i], x2, y2)
        D1[i] = [a1-a2, b1-b2, c1-c2, d1-d2]
    D1[:, 0] -= 1/n1
    dmin, dmax = -D1.min(), D1.max() + 1/n1
    return max(dmin, dmax)

def avgmaxdist(x1, y1, x2, y2):
    D1 = maxdist(x1, y1, x2, y2)
    D2 = maxdist(x2, y2, x1, y1)
    return (D1 + D2) / 2

def bks_distance_2d(x1, y1, x2, y2):
    """2D Bivariate KS distance (Fasano & Franceschini, 1987)."""
    n1, n2 = len(x1), len(x2)
    D = avgmaxdist(x1, y1, x2, y2)
    sqen = np.sqrt(n1 * n2 / (n1 + n2))
    r1 = pearsonr(x1, y1)[0]
    r2 = pearsonr(x2, y2)[0]
    rr = np.sqrt(1 - 0.5*(r1**2 + r2**2))
    prob = kstwobign.sf(D * sqen / (1 + rr*(0.25 - 0.75/sqen)))
    return D, prob

def compute_spi(D):
    return 100 * (1 - D)

print("BKS functions defined.")

---
## Cell 4 — MSN Implementation (Mitchell et al., 2024; Azzalini, 2005)

In [ ]:
def msn_pdf(y, xi, Omega, alpha):
    """Multivariate Skew-Normal PDF."""
    y = np.atleast_2d(y)
    xi, Omega, alpha = np.asarray(xi), np.asarray(Omega), np.asarray(alpha)
    try:
        Omega_inv = np.linalg.inv(Omega)
        phi_k = multivariate_normal.pdf(y, mean=xi, cov=Omega)
    except:
        return np.zeros(len(y))
    z = (y - xi) @ Omega_inv @ alpha
    return 2 * phi_k * norm.cdf(z)

def msn_log_likelihood(params, data):
    xi_x, xi_y, var_x, var_y, cov_xy, alpha_x, alpha_y = params
    if var_x <= 0 or var_y <= 0:
        return 1e10
    Omega = np.array([[var_x, cov_xy], [cov_xy, var_y]])
    if np.linalg.det(Omega) <= 0:
        return 1e10
    pdf_vals = msn_pdf(data, [xi_x, xi_y], Omega, [alpha_x, alpha_y])
    pdf_vals = np.clip(pdf_vals, 1e-10, None)
    return -np.sum(np.log(pdf_vals))

def fit_msn(data):
    """Fit Multivariate Skew-Normal to 2D data."""
    mx, my = data.mean(axis=0)
    vx, vy = data.var(axis=0)
    cov = np.cov(data.T)[0, 1]
    bounds = [(0,1),(0,1),(0.001,0.5),(0.001,0.5),(-0.2,0.2),(-10,10),(-10,10)]
    result = differential_evolution(
        msn_log_likelihood, bounds, args=(data,),
        seed=42, maxiter=500, tol=1e-6, workers=1
    )
    xi_x, xi_y, var_x, var_y, cov_xy, alpha_x, alpha_y = result.x
    Omega = np.array([[var_x, cov_xy], [cov_xy, var_y]])
    return {
        'xi': np.array([xi_x, xi_y]),
        'Omega': Omega,
        'alpha': np.array([alpha_x, alpha_y]),
        'skewness': skew(data),
        'success': result.success
    }

def sample_msn(xi, Omega, alpha, n=1000, seed=42):
    rng = np.random.default_rng(seed)
    k = len(xi)
    alpha = np.asarray(alpha)
    delta = alpha / np.sqrt(1 + alpha @ alpha)
    Omega_bar = Omega - np.outer(delta, delta) * np.diag(Omega)
    Omega_bar = (Omega_bar + Omega_bar.T) / 2 + np.eye(k) * 1e-8
    u0 = rng.normal(0, 1, n)
    u1 = rng.multivariate_normal(np.zeros(k), Omega_bar, n)
    samples = xi + np.outer(np.abs(u0), delta) + u1
    return samples

print("MSN functions defined.")

---
## Cell 5 — Load Data

In [ ]:
adhd_df        = pd.read_csv('ADHD_Group_XY_n30.csv')
autistic_df    = pd.read_csv('Autistic_Group_XY_n30.csv')
neurotypical_df = pd.read_csv('Neurotypical_Group_XY_n30.csv')

print("Loaded:")
for name, df in [('ADHD', adhd_df), ('Autistic', autistic_df), ('Neurotypical', neurotypical_df)]:
    print(f"  {name}: {len(df)} rows, {df['Participant Public ID'].nunique()} participants")

---
## Cell 6 — Prepare Data  ⚠ Y-axis inversion applied here

Gorilla's canvas has **Y = 0 at the top** (Eventful) and **Y = 1 at the bottom** (Uneventful).  
The ISO 12913 circumplex places Eventful at **+Y** (upward).  
Correction: `ISOEventful = 1 − Circumplex Y`  
ISOPleasant = Circumplex X (left = 0 Unpleasant, right = 1 Pleasant — already correct).

In [ ]:
def prepare_data(df):
    """
    Prepare dataframe for Soundscapy.

    Y-AXIS FIX: Gorilla records Y top-down (0=top/Eventful, 1=bottom/Uneventful).
    ISO circumplex is bottom-up (positive Y = Eventful).
    CALIBRATED: centre (0.516,0.507), arrowhead half-range 0.300/0.307.
    """
    df = df.copy()
    # CALIBRATED transform (from 4-point click test: centre + arrowheads).
    # True centre = (0.516, 0.507); half-range to arrowhead = 0.300 (P), 0.307 (E).
    # Kept in the [0,1] convention so downstream to_plot_scale ((v-0.5)*2)
    # yields calibrated centred coords reaching +/-1 at the arrowheads.
    df['ISOPleasant'] = 0.5 + (df['Circumplex X'] - 0.516) / 0.600   # 0.600 = 2*0.300
    df['ISOEventful'] = 0.5 + (0.507 - df['Circumplex Y']) / 0.614   # flip + scale; 0.614 = 2*0.307
    df = df.rename(columns={'Spreadsheet: Soundscape Name': 'Soundscape'})
    return df

adhd         = prepare_data(adhd_df)
autistic     = prepare_data(autistic_df)
neurotypical = prepare_data(neurotypical_df)
all_data     = pd.concat([adhd, autistic, neurotypical], ignore_index=True)

print("CALIBRATED transform applied (centre + arrowhead 4-point test)")
print(f"\nCoordinate ranges after correction:")
print(f"  ISOPleasant : [{all_data['ISOPleasant'].min():.3f}, {all_data['ISOPleasant'].max():.3f}]")
print(f"  ISOEventful : [{all_data['ISOEventful'].min():.3f}, {all_data['ISOEventful'].max():.3f}]")
print(f"\nTotal rows: {len(all_data)}")
print(all_data.groupby('Group')[['ISOPleasant','ISOEventful']].describe().round(3))

---
## Cell 7 — Summary Statistics by Group

In [ ]:
summary = all_data.groupby('Group').agg(
    n_participants=('Participant Public ID', 'nunique'),
    n_responses=('ISOPleasant', 'count'),
    Pleasant_mean=('ISOPleasant', 'mean'),
    Pleasant_std=('ISOPleasant', 'std'),
    Pleasant_median=('ISOPleasant', 'median'),
    Eventful_mean=('ISOEventful', 'mean'),
    Eventful_std=('ISOEventful', 'std'),
    Eventful_median=('ISOEventful', 'median')
).round(3)

print("Summary Statistics (after Y-inversion):")
display(summary)

---
## Cell 8 — Circumplex Scatter Plot: All Groups

In [ ]:
# Cell 8 — Scatter Plot with labels in corners

def to_plot_scale(df):
    d = df.copy()
    d['ISOPleasant'] = (d['ISOPleasant'] - 0.5) * 2
    d['ISOEventful'] = (d['ISOEventful'] - 0.5) * 2
    return d

adhd_plot         = to_plot_scale(adhd)
autistic_plot     = to_plot_scale(autistic)
neurotypical_plot = to_plot_scale(neurotypical)
all_data_plot     = to_plot_scale(all_data)

fig, ax = plt.subplots(figsize=(8, 8))

sspy.plotting.scatter(
    data=all_data_plot,
    x='ISOPleasant',
    y='ISOEventful',
    hue='Group',
    palette=GROUP_COLOURS,
    diagonal_lines=False,
    title='Soundscape Perception by Listener Group (n=30 per group)',
    xlim=(-1, 1),
    ylim=(-1, 1),
    s=50,
    alpha=0.6,
    ax=ax
)

# Draw diagonal lines manually
ax.plot([-1, 1], [-1, 1], color='grey', lw=1, ls='--', alpha=0.5)
ax.plot([-1, 1], [1, -1], color='grey', lw=1, ls='--', alpha=0.5)

# Quadrant labels — inside corners with white background
label_kw = dict(fontsize=10, color='#555555',
                transform=ax.transData,
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=2))

ax.text( 0.97,  0.97, 'Vibrant',    ha='right', va='top',    **label_kw)
ax.text(-0.97,  0.97, 'Chaotic',    ha='left',  va='top',    **label_kw)
ax.text(-0.97, -0.97, 'Monotonous', ha='left',  va='bottom', **label_kw)
ax.text( 0.97, -0.97, 'Calm',       ha='right', va='bottom', **label_kw)

plt.tight_layout()
plt.savefig('circumplex_scatter_all_groups.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: circumplex_scatter_all_groups.png")

## Response extremity by group (radial distance)

Unlocked by the calibration: pre-calibration every rating sat at r~0.6, so radius carried no information. Now r = sqrt(P^2 + E^2) measures how far from neutral each rating is. Tests whether groups differ in how *strongly/polarised* they rate, independent of direction.

In [ ]:
# Response extremity: radial distance from centre, by group
from scipy.stats import kruskal, mannwhitneyu

rad = all_data_plot.copy()                      # centred coords from to_plot_scale
rad['r'] = np.hypot(rad['ISOPleasant'], rad['ISOEventful'])

fig, ax = plt.subplots(figsize=(9, 5.5))
for g, col in GROUP_COLOURS.items():
    sns.kdeplot(rad.loc[rad['Group']==g, 'r'], ax=ax, color=col, fill=True,
                alpha=0.18, lw=2, label=g, clip=(0, 1.45), bw_adjust=0.9)
ax.axvline(1.0, color='grey', ls='--', lw=1, alpha=0.6)
ax.text(1.0, ax.get_ylim()[1]*0.96, ' rim (arrowhead = max)', color='grey', fontsize=9, va='top')
ax.set_xlabel(r'Radial distance from centre  $r=\sqrt{P_{ISO}^2 + E_{ISO}^2}$')
ax.set_ylabel('Density')
ax.set_title('Response extremity by listener group')
ax.legend(title='Group', frameon=False)
plt.tight_layout()
plt.savefig('radial_distribution_by_group.png', dpi=150, bbox_inches='tight')
plt.show()

print(rad.groupby('Group')['r'].agg(['median','mean','std']).round(3))
groups = [rad.loc[rad['Group']==g, 'r'] for g in ['ADHD','Autistic','Neurotypical']]
H, p = kruskal(*groups)
print(f'\nKruskal-Wallis (r across 3 groups): H={H:.3f}, p={p:.4f}')
for a in ['ADHD','Autistic']:
    U, pu = mannwhitneyu(rad.loc[rad['Group']==a,'r'],
                         rad.loc[rad['Group']=='Neurotypical','r'], alternative='two-sided')
    print(f'  {a} vs Neurotypical: U={U:.0f}, p={pu:.4f}')


---
## Cell 9 — Density Plots: Each Group Separately

In [ ]:
# Cell 9 — Density Plots: Each Group Separately

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (group, color, df) in zip(axes, [
    ('ADHD',         '#E69F00', adhd_plot),
    ('Autistic',     '#56B4E9', autistic_plot),
    ('Neurotypical', '#009E73', neurotypical_plot)
]):
    sspy.plotting.density(
        data=df,
        x='ISOPleasant',
        y='ISOEventful',
        title=f'{group} (n={df["Participant Public ID"].nunique()})',
        color=color,
        diagonal_lines=False,
        xlim=(-1, 1),
        ylim=(-1, 1),
        incl_scatter=True,
        ax=ax
    )
    ax.plot([-1,1],[-1,1], color='grey', lw=1, ls='--', alpha=0.5)
    ax.plot([-1,1],[1,-1], color='grey', lw=1, ls='--', alpha=0.5)
    label_kw = dict(fontsize=8, color='#555555', transform=ax.transData,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1))
    ax.text( 0.97,  0.97, 'Vibrant',    ha='right', va='top',    **label_kw)
    ax.text(-0.97,  0.97, 'Chaotic',    ha='left',  va='top',    **label_kw)
    ax.text(-0.97, -0.97, 'Monotonous', ha='left',  va='bottom', **label_kw)
    ax.text( 0.97, -0.97, 'Calm',       ha='right', va='bottom', **label_kw)

plt.suptitle('Circumplex Density — All Groups', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('circumplex_density_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: circumplex_density_by_group.png")

---
## Cell 10 — Density Plot: Groups Overlaid

In [ ]:
# Cell 10 — Density Plot: Groups Overlaid

fig, ax = plt.subplots(figsize=(8, 8))

sspy.plotting.density(
    data=all_data_plot,
    x='ISOPleasant',
    y='ISOEventful',
    hue='Group',
    palette=GROUP_COLOURS,
    title='Soundscape Perception Density — All Groups Overlaid',
    diagonal_lines=False,
    xlim=(-1, 1),
    ylim=(-1, 1),
    incl_scatter=False,
    alpha=0.5,
    legend=True,
    ax=ax
)

ax.plot([-1,1],[-1,1], color='grey', lw=1, ls='--', alpha=0.5)
ax.plot([-1,1],[1,-1], color='grey', lw=1, ls='--', alpha=0.5)
label_kw = dict(fontsize=10, color='#555555', transform=ax.transData,
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=2))
ax.text( 0.97,  0.97, 'Vibrant',    ha='right', va='top',    **label_kw)
ax.text(-0.97,  0.97, 'Chaotic',    ha='left',  va='top',    **label_kw)
ax.text(-0.97, -0.97, 'Monotonous', ha='left',  va='bottom', **label_kw)
ax.text( 0.97, -0.97, 'Calm',       ha='right', va='bottom', **label_kw)

plt.tight_layout()
plt.savefig('circumplex_density_overlaid.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: circumplex_density_overlaid.png")

---
## Cell 11 — Per-Soundscape Faceted Circumplex Plots

In [ ]:
# Cell 11 — Per-Soundscape Faceted Circumplex Plots

soundscapes = sorted(all_data_plot['Soundscape'].unique())
ncols = 4
nrows = int(np.ceil(len(soundscapes) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4*nrows))
axes = axes.flatten()

for idx, soundscape in enumerate(soundscapes):
    ax = axes[idx]
    subset = all_data_plot[all_data_plot['Soundscape'] == soundscape]
    sspy.plotting.scatter(
        data=subset,
        x='ISOPleasant',
        y='ISOEventful',
        hue='Group',
        palette=GROUP_COLOURS,
        diagonal_lines=False,
        title=soundscape,
        xlim=(-1, 1),
        ylim=(-1, 1),
        s=40,
        alpha=0.7,
        legend=(idx == 0),
        ax=ax
    )
    ax.plot([-1,1],[-1,1], color='grey', lw=0.8, ls='--', alpha=0.4)
    ax.plot([-1,1],[1,-1], color='grey', lw=0.8, ls='--', alpha=0.4)
    label_kw = dict(fontsize=7, color='#666666', transform=ax.transData,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1))
    ax.text( 0.95,  0.95, 'V',  ha='right', va='top',    **label_kw)
    ax.text(-0.95,  0.95, 'Ch', ha='left',  va='top',    **label_kw)
    ax.text(-0.95, -0.95, 'M',  ha='left',  va='bottom', **label_kw)
    ax.text( 0.95, -0.95, 'Ca', ha='right', va='bottom', **label_kw)

for idx in range(len(soundscapes), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Circumplex by Soundscape — All Groups', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('circumplex_by_soundscape.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: circumplex_by_soundscape.png")

---
## Cell 12 — Gaussian KDE Heatmaps

In [ ]:
# Cell 12 — Gaussian KDE Heatmaps

from scipy.stats import gaussian_kde

def draw_circumplex_axes(ax):
    ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
    ax.axvline(0, color='k', lw=0.8, ls='--', alpha=0.4)
    label_kw = dict(fontsize=8, color='#555555', transform=ax.transData,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1))
    ax.text( 0.95,  0.95, 'Vibrant',    ha='right', va='top',    **label_kw)
    ax.text(-0.95,  0.95, 'Chaotic',    ha='left',  va='top',    **label_kw)
    ax.text(-0.95, -0.95, 'Monotonous', ha='left',  va='bottom', **label_kw)
    ax.text( 0.95, -0.95, 'Calm',       ha='right', va='bottom', **label_kw)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (df, name, cmap) in zip(axes, [
    (adhd_plot,         'ADHD',         'YlOrRd'),
    (autistic_plot,     'Autistic',     'YlGnBu'),
    (neurotypical_plot, 'Neurotypical', 'YlGn')
]):
    x, y = df['ISOPleasant'].values, df['ISOEventful'].values
    xi = yi = np.linspace(-1, 1, 100)
    Xi, Yi = np.meshgrid(xi, yi)
    kernel = gaussian_kde(np.vstack([x, y]))
    Zi = kernel(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)
    ax.imshow(Zi, origin='lower', extent=[-1,1,-1,1], cmap=cmap, aspect='equal')
    ax.scatter(x, y, c='white', s=8, alpha=0.4)
    draw_circumplex_axes(ax)
    ax.plot([-1,1],[-1,1], color='white', lw=0.8, ls='--', alpha=0.4)
    ax.plot([-1,1],[1,-1], color='white', lw=0.8, ls='--', alpha=0.4)
    ax.set_title(f'{name} (n={df["Participant Public ID"].nunique()})', fontsize=11)
    ax.set_xlabel('ISOPleasant')
    ax.set_ylabel('ISOEventful')

plt.suptitle('Gaussian KDE Density Heatmaps', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('Heatmap_Combined.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: Heatmap_Combined.png")

---
## Cell 13 — Per-Soundscape Mean Coordinates Table

In [ ]:
# Cell 13 — Per-Soundscape Mean Coordinates Table

soundscape_means = all_data.groupby(['Soundscape', 'Group']).agg(
    ISOPleasant=('ISOPleasant', 'mean'),
    ISOEventful=('ISOEventful', 'mean'),
    n=('ISOPleasant', 'count')
).round(3).reset_index()

pivot_p = soundscape_means.pivot(index='Soundscape', columns='Group', values='ISOPleasant')
pivot_e = soundscape_means.pivot(index='Soundscape', columns='Group', values='ISOEventful')

print("Mean ISOPleasant by Soundscape × Group:")
display(pivot_p)
print("\nMean ISOEventful by Soundscape × Group (Y-corrected):")
display(pivot_e)

---
## Cell 14 — Heatmaps: Pleasant & Eventful Mean Scores

In [ ]:
# Cell 14 — Heatmaps: Pleasant & Eventful Mean Scores

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(pivot_p, annot=True, fmt='.3f', cmap='RdYlGn',
            center=0.5, vmin=0, vmax=1, ax=axes[0])
axes[0].set_title('Mean ISOPleasant by Soundscape × Group', fontsize=12)

sns.heatmap(pivot_e, annot=True, fmt='.3f', cmap='RdYlBu_r',
            center=0.5, vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('Mean ISOEventful by Soundscape × Group', fontsize=12)

plt.tight_layout()
plt.savefig('heatmap_means.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: heatmap_means.png")

---
## Cell 15 — Violin Plots: ISOPleasant & ISOEventful Distributions

In [ ]:
# Cell 15 — Violin Plots: ISOPleasant & ISOEventful Distributions

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, title in [
    (axes[0], 'ISOPleasant', 'ISOPleasant Distribution by Group'),
    (axes[1], 'ISOEventful', 'ISOEventful Distribution by Group (Y-corrected)')
]:
    sns.violinplot(data=all_data, x='Group', y=col, palette=GROUP_COLOURS, ax=ax)
    ax.set_title(title)
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color='k', ls='--', lw=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig('violin_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: violin_distributions.png")

---
## Cell 16 — Quadrant Distribution
Quadrants defined relative to centre (0.5, 0.5) in [0,1] space.

In [ ]:
# Cell 16 — Quadrant Distribution

def quadrant_label(row, cx=0.5, cy=0.5):
    p, e = row['ISOPleasant'], row['ISOEventful']
    if   p >= cx and e >= cy: return 'Vibrant'
    elif p <  cx and e >= cy: return 'Chaotic'
    elif p <  cx and e <  cy: return 'Monotonous'
    else:                     return 'Calm'

all_data['Quadrant'] = all_data.apply(quadrant_label, axis=1)

quad_pct = (
    all_data.groupby(['Group','Quadrant']).size()
    .unstack(fill_value=0)
    .apply(lambda r: 100*r/r.sum(), axis=1)
    .round(1)
)

print("Quadrant distribution (%):")
display(quad_pct)

fig, ax = plt.subplots(figsize=(10, 5))
quad_pct[['Vibrant','Chaotic','Monotonous','Calm']].plot(
    kind='bar', ax=ax,
    color=['#2ecc71','#e74c3c','#95a5a6','#3498db'],
    edgecolor='black', linewidth=0.5
)
ax.set_title('Quadrant Distribution by Group', fontsize=13)
ax.set_xlabel('Group')
ax.set_ylabel('Percentage of Responses (%)')
ax.legend(title='Quadrant')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('quadrant_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: quadrant_distribution.png")

---
## Cell 17 — Fit MSN Distributions (Mitchell et al., 2024)

In [ ]:
# Cell 17 — Fit MSN Distributions (Mitchell et al., 2024)
# ⚠ Takes 2-5 minutes — leave running

print("Fitting Multivariate Skew-Normal distributions...\n")

adhd_data         = adhd[['ISOPleasant','ISOEventful']].values
autistic_data     = autistic[['ISOPleasant','ISOEventful']].values
neurotypical_data = neurotypical[['ISOPleasant','ISOEventful']].values

msn_adhd         = fit_msn(adhd_data)
msn_autistic     = fit_msn(autistic_data)
msn_neurotypical = fit_msn(neurotypical_data)

msn_fits = {'ADHD': msn_adhd, 'Autistic': msn_autistic, 'Neurotypical': msn_neurotypical}

for name, fit in msn_fits.items():
    print(f"{name}:  xi=[{fit['xi'][0]:.4f}, {fit['xi'][1]:.4f}]  "
          f"alpha=[{fit['alpha'][0]:.4f}, {fit['alpha'][1]:.4f}]  "
          f"skew=[{fit['skewness'][0]:.4f}, {fit['skewness'][1]:.4f}]")

print("\nMSN fitting complete!")

---
## Cell 18 — MSN Parameters Summary Table

In [ ]:
# Cell 18 — MSN Parameters Summary Table

msn_summary = []
for name, fit in msn_fits.items():
    msn_summary.append({
        'Group':          name,
        'xi_Pleasant':    fit['xi'][0],
        'xi_Eventful':    fit['xi'][1],
        'Var_Pleasant':   fit['Omega'][0,0],
        'Var_Eventful':   fit['Omega'][1,1],
        'Cov_PE':         fit['Omega'][0,1],
        'alpha_Pleasant': fit['alpha'][0],
        'alpha_Eventful': fit['alpha'][1],
        'Skew_P':         fit['skewness'][0],
        'Skew_E':         fit['skewness'][1]
    })

msn_df = pd.DataFrame(msn_summary)
print("MSN Parameters")
display(msn_df.round(4))

---
## Cell 19 — MSN Fitted Distribution Plots

In [ ]:
# Cell 19 — MSN Fitted Distribution Plots

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

adhd_plot_arr         = adhd_plot[['ISOPleasant','ISOEventful']].values
autistic_plot_arr     = autistic_plot[['ISOPleasant','ISOEventful']].values
neurotypical_plot_arr = neurotypical_plot[['ISOPleasant','ISOEventful']].values

for ax, (name, color, data, fit) in zip(axes, [
    ('ADHD',         '#E69F00', adhd_plot_arr,         msn_adhd),
    ('Autistic',     '#56B4E9', autistic_plot_arr,     msn_autistic),
    ('Neurotypical', '#009E73', neurotypical_plot_arr, msn_neurotypical)
]):
    xi_plot = [(fit['xi'][0] - 0.5)*2, (fit['xi'][1] - 0.5)*2]

    xi_g = yi_g = np.linspace(-1, 1, 80)
    Xi, Yi = np.meshgrid(xi_g, yi_g)
    Xi01 = Xi/2 + 0.5
    Yi01 = Yi/2 + 0.5
    positions = np.column_stack([Xi01.ravel(), Yi01.ravel()])
    Zi = msn_pdf(positions, fit['xi'], fit['Omega'], fit['alpha']).reshape(Xi.shape)

    ax.scatter(data[:,0], data[:,1], c=color, alpha=0.4, s=20)
    ax.contour(Xi, Yi, Zi, levels=6, colors='black', linewidths=1, alpha=0.7)
    ax.contourf(Xi, Yi, Zi, levels=20, cmap='Greys', alpha=0.3)
    ax.scatter(xi_plot[0], xi_plot[1], c='red', s=120, marker='*', zorder=5)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
    ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.4)
    ax.axvline(0, color='k', lw=0.5, ls='--', alpha=0.4)
    ax.plot([-1,1],[-1,1], color='grey', lw=0.8, ls='--', alpha=0.4)
    ax.plot([-1,1],[1,-1], color='grey', lw=0.8, ls='--', alpha=0.4)
    ax.set_xlabel('ISOPleasant')
    ax.set_ylabel('ISOEventful')
    label_kw = dict(fontsize=8, color='#555555', transform=ax.transData,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1))
    ax.text( 0.95,  0.95, 'Vibrant',    ha='right', va='top',    **label_kw)
    ax.text(-0.95,  0.95, 'Chaotic',    ha='left',  va='top',    **label_kw)
    ax.text(-0.95, -0.95, 'Monotonous', ha='left',  va='bottom', **label_kw)
    ax.text( 0.95, -0.95, 'Calm',       ha='right', va='bottom', **label_kw)
    ax.set_title(f'{name}\nxi=[{xi_plot[0]:.3f}, {xi_plot[1]:.3f}]')

plt.suptitle('Fitted MSN Distributions by Group', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('MSN_fits_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: MSN_fits_by_group.png")

---
## Cell 20 — Compute SPI Tables (per soundscape)

In [ ]:
# Cell 20 — Compute SPI Tables (per soundscape)

def compute_spi_table(df1, df2, g1, g2):
    soundscapes = sorted(set(df1['Soundscape'].unique()) & set(df2['Soundscape'].unique()))
    results = []
    for ss in soundscapes:
        s1 = df1[df1['Soundscape'] == ss]
        s2 = df2[df2['Soundscape'] == ss]
        x1, y1 = s1['ISOPleasant'].values, s1['ISOEventful'].values
        x2, y2 = s2['ISOPleasant'].values, s2['ISOEventful'].values
        if len(x1) >= 3 and len(x2) >= 3:
            D, p = bks_distance_2d(x1, y1, x2, y2)
            results.append({
                'Soundscape': ss,
                f'n_{g1}': len(x1),
                f'n_{g2}': len(x2),
                'D_BKS': round(D, 4),
                'p_value': round(p, 4),
                'SPI': round(compute_spi(D), 2)
            })
    return pd.DataFrame(results)

spi_adhd_nt  = compute_spi_table(adhd, neurotypical, 'ADHD', 'NT')
spi_aut_nt   = compute_spi_table(autistic, neurotypical, 'Autistic', 'NT')
spi_adhd_aut = compute_spi_table(adhd, autistic, 'ADHD', 'Autistic')

print("SPI: ADHD vs Neurotypical")
display(spi_adhd_nt)
print("\nSPI: Autistic vs Neurotypical")
display(spi_aut_nt)
print("\nSPI: ADHD vs Autistic")
display(spi_adhd_aut)

---
## Cell 21 — Overall SPI (All Soundscapes Pooled)

In [ ]:
# Cell 21 — Overall SPI (All Soundscapes Pooled)

print("Overall SPI — all soundscapes pooled:\n")
for label, d1, d2 in [
    ('ADHD vs Neurotypical',    adhd, neurotypical),
    ('Autistic vs Neurotypical', autistic, neurotypical),
    ('ADHD vs Autistic',         adhd, autistic)
]:
    D, p = bks_distance_2d(
        d1['ISOPleasant'].values, d1['ISOEventful'].values,
        d2['ISOPleasant'].values, d2['ISOEventful'].values
    )
    print(f"  {label:30s}  D_BKS={D:.4f}  p={p:.4f}  SPI={compute_spi(D):.2f}")

---
## Cell 22 — SPI Bar Chart

In [ ]:
# Cell 22 — SPI Bar Chart

fig, ax = plt.subplots(figsize=(13, 6))

ss_labels = spi_adhd_nt['Soundscape'].values
x = np.arange(len(ss_labels))
w = 0.25

ax.bar(x - w, spi_adhd_nt['SPI'],  w, label='ADHD vs NT',      color='#E69F00')
ax.bar(x,     spi_aut_nt['SPI'],   w, label='Autistic vs NT',   color='#56B4E9')
ax.bar(x + w, spi_adhd_aut['SPI'], w, label='ADHD vs Autistic', color='#009E73')

ax.axhline(90, color='#CC0000', ls='--', lw=1.5, alpha=0.8, label='SPI=90')
ax.axhline(75, color='#555555', ls='--', lw=1.5, alpha=0.8, label='SPI=75')
ax.set_xticks(x)
ax.set_xticklabels(ss_labels, rotation=35, ha='right')
ax.set_ylabel('SPI Score (0–100)')
ax.set_xlabel('Soundscape')
ax.set_title('Soundscape Perception Index by Soundscape and Group Pair', fontsize=13)
ax.set_ylim(0, 105)
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), frameon=True)

plt.tight_layout()
plt.savefig('SPI_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: SPI_bar_chart.png")

---
## Cell 23 — Statistical Tests (Kruskal-Wallis + Post-hoc)

In [ ]:
# Cell 23 — Statistical Tests (Kruskal-Wallis + Mann-Whitney post-hoc)

from scipy.stats import kruskal, mannwhitneyu

print("Kruskal-Wallis (3-group comparison):\n")
for col in ['ISOPleasant', 'ISOEventful']:
    H, p = kruskal(
        adhd[col].dropna(),
        autistic[col].dropna(),
        neurotypical[col].dropna()
    )
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"  {col}: H={H:.3f}, p={p:.4f} {sig}")

print("\nMann-Whitney U post-hoc (pairwise, uncorrected):\n")
pairs = [
    ('ADHD', 'Neurotypical', adhd, neurotypical),
    ('Autistic', 'Neurotypical', autistic, neurotypical),
    ('ADHD', 'Autistic', adhd, autistic)
]
for g1, g2, d1, d2 in pairs:
    for col in ['ISOPleasant', 'ISOEventful']:
        U, p = mannwhitneyu(d1[col].dropna(), d2[col].dropna(),
                            alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f"  {g1} vs {g2} | {col}: U={U:.0f}, p={p:.4f} {sig}")
    print()

---
## Cell 24 — Export All Results

In [ ]:
# Cell 24 — Export All Results

spi_adhd_nt.to_csv('SPI_ADHD_vs_NT.csv', index=False)
spi_aut_nt.to_csv('SPI_Autistic_vs_NT.csv', index=False)
spi_adhd_aut.to_csv('SPI_ADHD_vs_Autistic.csv', index=False)
msn_df.to_csv('MSN_Parameters.csv', index=False)
soundscape_means.to_csv('Soundscape_Means_by_Group.csv', index=False)
quad_pct.to_csv('Quadrant_Distribution.csv')
all_data.to_csv('all_data_corrected.csv', index=False)

print("Exports complete:")
for f in ['SPI_ADHD_vs_NT.csv', 'SPI_Autistic_vs_NT.csv',
          'SPI_ADHD_vs_Autistic.csv', 'MSN_Parameters.csv',
          'Soundscape_Means_by_Group.csv', 'Quadrant_Distribution.csv',
          'all_data_corrected.csv']:
    print(f"  ✓ {f}")

---
## Summary

### Key fix applied
| Column | Source | Transformation |
|--------|--------|----------------|
| ISOPleasant | Circumplex X | None — X=0 Unpleasant → X=1 Pleasant ✓ |
| ISOEventful | Circumplex Y | **1 − Y** — Gorilla Y=0 is top (Eventful), ISO Y=1 is Eventful |

### Dataset
- **n = 30 per group** (90 total)
- ADHD: 2 excluded (bypassed circumplex screen) + 2 excluded (lowest completeness)
- Autistic: 4 excluded (no task data)
- Neurotypical: 0 excluded

### Output files
**SPI:** `SPI_ADHD_vs_NT.csv`, `SPI_Autistic_vs_NT.csv`, `SPI_ADHD_vs_Autistic.csv`  
**MSN:** `MSN_Parameters.csv`  
**Tables:** `Soundscape_Means_by_Group.csv`, `Quadrant_Distribution.csv`  
**Data:** `all_data_corrected.csv`  
**Plots:** `circumplex_scatter_all_groups.png`, `circumplex_density_by_group.png`,  
`circumplex_density_overlaid.png`, `circumplex_by_soundscape.png`,  
`Heatmap_Combined.png`, `heatmap_means.png`, `violin_distributions.png`,  
`quadrant_distribution.png`, `MSN_fits_by_group.png`, `SPI_bar_chart.png`

In [ ]:
# Cell 25 — Per-Group Per-Soundscape Scatter

for group, color, df in [
    ('ADHD',         '#E69F00', adhd_plot),
    ('Autistic',     '#56B4E9', autistic_plot),
    ('Neurotypical', '#009E73', neurotypical_plot)
]:
    soundscapes = sorted(df['Soundscape'].unique())
    ncols = 5
    nrows = int(np.ceil(len(soundscapes) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, 4*nrows))
    axes = axes.flatten()

    for idx, soundscape in enumerate(soundscapes):
        ax = axes[idx]
        subset = df[df['Soundscape'] == soundscape]
        ax.scatter(subset['ISOPleasant'], subset['ISOEventful'],
                   c=color, alpha=0.7, s=40)
        ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
        ax.axvline(0, color='k', lw=0.8, ls='--', alpha=0.4)
        ax.plot([-1,1],[-1,1], color='grey', lw=0.8, ls='--', alpha=0.4)
        ax.plot([-1,1],[1,-1], color='grey', lw=0.8, ls='--', alpha=0.4)
        ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
        ax.set_title(soundscape, fontsize=9)
        ax.set_xlabel('P_ISO', fontsize=8)
        ax.set_ylabel('E_ISO', fontsize=8)
        label_kw = dict(fontsize=6, color='#666666', transform=ax.transData,
                        bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1))
        ax.text( 0.95,  0.95, 'V',  ha='right', va='top',    **label_kw)
        ax.text(-0.95,  0.95, 'Ch', ha='left',  va='top',    **label_kw)
        ax.text(-0.95, -0.95, 'M',  ha='left',  va='bottom', **label_kw)
        ax.text( 0.95, -0.95, 'Ca', ha='right', va='bottom', **label_kw)

    for idx in range(len(soundscapes), len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle(f'{group} Group — Responses by Soundscape', fontsize=13, y=1.01)
    plt.tight_layout()
    fname = f'{group}_by_soundscape.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

In [ ]:
# Cell 26 — MSN Scatter with Contours by Group (publication style)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

adhd_plot_arr         = adhd_plot[['ISOPleasant','ISOEventful']].values
autistic_plot_arr     = autistic_plot[['ISOPleasant','ISOEventful']].values
neurotypical_plot_arr = neurotypical_plot[['ISOPleasant','ISOEventful']].values

for ax, (name, color, data, fit) in zip(axes, [
    ('ADHD',         '#E69F00', adhd_plot_arr,         msn_adhd),
    ('Autistic',     '#56B4E9', autistic_plot_arr,     msn_autistic),
    ('Neurotypical', '#009E73', neurotypical_plot_arr, msn_neurotypical)
]):
    xi_plot = [(fit['xi'][0] - 0.5)*2, (fit['xi'][1] - 0.5)*2]
    xi_g = yi_g = np.linspace(-1, 1, 80)
    Xi, Yi = np.meshgrid(xi_g, yi_g)
    positions = np.column_stack([(Xi/2+0.5).ravel(), (Yi/2+0.5).ravel()])
    Zi = msn_pdf(positions, fit['xi'], fit['Omega'], fit['alpha']).reshape(Xi.shape)

    ax.scatter(data[:,0], data[:,1], c=color, alpha=0.5, s=20)
    ax.contour(Xi, Yi, Zi, levels=5, colors='black', linewidths=1, alpha=0.8)
    ax.plot(xi_plot[0], xi_plot[1], 'rx', ms=12, mew=2,
            label=f'ξ = ({xi_plot[0]:.2f}, {xi_plot[1]:.2f})')
    ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.4)
    ax.axvline(0, color='k', lw=0.5, ls='--', alpha=0.4)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
    ax.set_xlabel('P_ISO'); ax.set_ylabel('E_ISO')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_title(f'{name} (n={len(data)})', fontsize=11)
    label_kw = dict(fontsize=8, color='#555555', transform=ax.transData,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1))
    ax.text( 0.95,  0.95, 'Vibrant',    ha='right', va='top',    **label_kw)
    ax.text(-0.95,  0.95, 'Chaotic',    ha='left',  va='top',    **label_kw)
    ax.text(-0.95, -0.95, 'Monotonous', ha='left',  va='bottom', **label_kw)
    ax.text( 0.95, -0.95, 'Calm',       ha='right', va='bottom', **label_kw)

plt.suptitle('MSN Distribution Fits by Group', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('MSN_density_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: MSN_density_by_group.png")

In [ ]:
# Cell 27 — MSN Fits Per Soundscape by Group

for group, color, df, fit_group in [
    ('ADHD',         '#E69F00', adhd,         msn_adhd),
    ('Autistic',     '#56B4E9', autistic,     msn_autistic),
    ('Neurotypical', '#009E73', neurotypical, msn_neurotypical)
]:
    soundscapes = sorted(df['Soundscape'].unique())
    ncols = 5
    nrows = int(np.ceil(len(soundscapes) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, 4*nrows))
    axes = axes.flatten()

    for idx, soundscape in enumerate(soundscapes):
        ax = axes[idx]
        subset = df[df['Soundscape'] == soundscape]
        data_ss = subset[['ISOPleasant','ISOEventful']].values

        if len(data_ss) >= 5:
            try:
                fit_ss = fit_msn(data_ss)
                xi_plot = [(fit_ss['xi'][0]-0.5)*2, (fit_ss['xi'][1]-0.5)*2]
                xi_g = yi_g = np.linspace(-1, 1, 60)
                Xi, Yi = np.meshgrid(xi_g, yi_g)
                positions = np.column_stack([(Xi/2+0.5).ravel(),(Yi/2+0.5).ravel()])
                Zi = msn_pdf(positions, fit_ss['xi'], fit_ss['Omega'],
                             fit_ss['alpha']).reshape(Xi.shape)
                ax.contour(Xi, Yi, Zi, levels=4, colors='black', linewidths=0.8, alpha=0.8)
                ax.plot(xi_plot[0], xi_plot[1], 'rx', ms=10, mew=2)
            except:
                pass

        # Plot scatter in [-1,1] space
        x_plot = (data_ss[:,0] - 0.5)*2
        y_plot = (data_ss[:,1] - 0.5)*2
        ax.scatter(x_plot, y_plot, c=color, alpha=0.6, s=25)
        ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.4)
        ax.axvline(0, color='k', lw=0.5, ls='--', alpha=0.4)
        ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
        ax.set_title(soundscape, fontsize=8)
        ax.set_xlabel('P_ISO', fontsize=7)
        ax.set_ylabel('E_ISO', fontsize=7)

    for idx in range(len(soundscapes), len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle(f'{group} — MSN Fits by Soundscape', fontsize=13, y=1.01)
    plt.tight_layout()
    fname = f'MSN_{group}_by_soundscape.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

In [ ]:
# Cell 28 — SPI Full Reference Table (ADHD as reference, SPI_ADHD = 100)

rows = []
soundscapes = sorted(set(adhd['Soundscape'].unique()) &
                     set(autistic['Soundscape'].unique()) &
                     set(neurotypical['Soundscape'].unique()))

for ss in soundscapes:
    a  = adhd[adhd['Soundscape']==ss]
    au = autistic[autistic['Soundscape']==ss]
    nt = neurotypical[neurotypical['Soundscape']==ss]

    x_a,  y_a  = a['ISOPleasant'].values,  a['ISOEventful'].values
    x_au, y_au = au['ISOPleasant'].values, au['ISOEventful'].values
    x_nt, y_nt = nt['ISOPleasant'].values, nt['ISOEventful'].values

    D_nt_vs_a,  _ = bks_distance_2d(x_nt, y_nt, x_a, y_a)
    D_au_vs_a,  _ = bks_distance_2d(x_au, y_au, x_a, y_a)

    rows.append({
        'Soundscape':      ss,
        'N_ADHD':          len(a),
        'N_NT':            len(nt),
        'N_Autistic':      len(au),
        'D_NT_vs_ADHD':    round(D_nt_vs_a, 3),
        'D_Aut_vs_ADHD':   round(D_au_vs_a, 3),
        'SPI_ADHD':        100,
        'SPI_NT_vs_ADHD':  round(compute_spi(D_nt_vs_a), 1),
        'SPI_Aut_vs_ADHD': round(compute_spi(D_au_vs_a), 1)
    })

spi_adhd_ref = pd.DataFrame(rows)
print("SPI Full Reference Table (ADHD as reference):")
display(spi_adhd_ref)
spi_adhd_ref.to_csv('SPI_Full_ADHDReference_Table.csv', index=False)
print("Saved: SPI_Full_ADHDReference_Table.csv")

In [ ]:
# Cell 29 — Participant Trajectories by Group

soundscape_order = sorted(adhd['Soundscape'].unique())

for group, df in [('ADHD', adhd), ('Autistic', autistic), ('Neurotypical', neurotypical)]:
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))

    for ax, col, ylabel, title in [
        (axes[0], 'ISOEventful', 'ISOEventful',
         f'Participant Trajectories: Eventful ({group} Group)'),
        (axes[1], 'ISOPleasant', 'ISOPleasant',
         f'Participant Trajectories: Pleasant ({group} Group)')
    ]:
        participants = df['Participant Public ID'].unique()
        for pid in participants:
            pdata = df[df['Participant Public ID']==pid].set_index('Soundscape')
            vals = [pdata.loc[ss, col] if ss in pdata.index else np.nan
                    for ss in soundscape_order]
            ax.plot(soundscape_order, vals, marker='o', ms=5, alpha=0.6, lw=1)

        ax.set_xticks(range(len(soundscape_order)))
        ax.set_xticklabels(soundscape_order, rotation=35, ha='right', fontsize=9)
        ax.set_ylabel(ylabel)
        ax.set_ylim(0, 1)
        ax.axhline(0.5, color='k', ls='--', lw=0.8, alpha=0.4)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fname = f'trajectories_{group}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

In [ ]:
# Cell 30 — Triangulation Scatter by Soundscape (All Groups)

soundscapes = sorted(all_data_plot['Soundscape'].unique())
ncols = 5
nrows = int(np.ceil(len(soundscapes) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 4*nrows))
axes = axes.flatten()

for idx, soundscape in enumerate(soundscapes):
    ax = axes[idx]
    subset = all_data_plot[all_data_plot['Soundscape'] == soundscape]

    for group, color, marker in [
        ('ADHD',         '#E69F00', 'o'),
        ('Autistic',     '#56B4E9', 's'),
        ('Neurotypical', '#009E73', '^')
    ]:
        g = subset[subset['Group'] == group]
        ax.scatter(g['ISOPleasant'], g['ISOEventful'],
                   c=color, marker=marker, alpha=0.7, s=30,
                   label=group if idx == 0 else '')

    ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.4)
    ax.axvline(0, color='k', lw=0.5, ls='--', alpha=0.4)
    ax.plot([-1,1],[-1,1], color='grey', lw=0.6, ls='--', alpha=0.3)
    ax.plot([-1,1],[1,-1], color='grey', lw=0.6, ls='--', alpha=0.3)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
    ax.set_title(soundscape, fontsize=9)
    ax.set_xlabel('P_ISO', fontsize=8)
    ax.set_ylabel('E_ISO', fontsize=8)

    if idx == 0:
        ax.legend(fontsize=7, loc='lower left')

for idx in range(len(soundscapes), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Circumplex by Soundscape (All Groups)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('TRIANGULATION_scatter_by_soundscape.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: TRIANGULATION_scatter_by_soundscape.png")

In [ ]:
# Diagnostic — run this before any more plotting attempts
print("=== ADHD ===")
print(adhd[['Circumplex X', 'Circumplex Y', 'ISOPleasant', 'ISOEventful']].describe().round(3))
print()
print("=== First 5 rows ===")
print(adhd[['Circumplex X', 'Circumplex Y', 'ISOPleasant', 'ISOEventful']].head())

In [ ]:
# Cell 8b — Uncorrected scatter — matching original exactly
import pandas as pd
import matplotlib.pyplot as plt

GROUP_COLORS = {
    'ADHD':         '#1f77b4',
    'Autistic':     '#ff7f0e',
    'Neurotypical': '#2ca02c',
}

frames = []
for group, df in [('ADHD', adhd), ('Autistic', autistic), ('Neurotypical', neurotypical)]:
    tmp = df.copy()
    tmp['ISOEventful'] = tmp['Circumplex Y']
    tmp['Group'] = group
    frames.append(tmp)

combined = pd.concat(frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 10))

for group, color in GROUP_COLORS.items():
    subset = combined[combined['Group'] == group]
    ax.scatter(
        2 * subset['ISOPleasant'] - 1,
        2 * subset['ISOEventful'] - 1,
        color=color, alpha=0.6, s=20, label=group, zorder=3
    )

ax.axhline(0, color='#555555', linewidth=0.8, linestyle='--', zorder=2)
ax.axvline(0, color='#555555', linewidth=0.8, linestyle='--', zorder=2)

ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_xlabel('$P_{ISO}$', fontsize=13)
ax.set_ylabel('$E_{ISO}$', fontsize=13)
ax.set_title('Soundscape Perception by Group', fontsize=15)

ax.set_xticks([-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1])
ax.set_yticks([-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1])
ax.minorticks_on()
ax.xaxis.set_minor_locator(plt.MultipleLocator(0.05))
ax.yaxis.set_minor_locator(plt.MultipleLocator(0.05))
ax.grid(True, which='major', color='#9999bb', linewidth=1.0, linestyle=':', alpha=1.0)
ax.grid(True, which='minor', color='#9999bb', linewidth=0.5, linestyle=':', alpha=1.0)
ax.set_axisbelow(True)

ax.text(-0.98,  0.98, 'Chaotic',    fontsize=9, color='#666666', ha='left',  va='top')
ax.text( 0.98,  0.98, 'Vibrant',    fontsize=9, color='#666666', ha='right', va='top')
ax.text(-0.98, -0.98, 'Monotonous', fontsize=9, color='#666666', ha='left',  va='bottom')
ax.text( 0.98, -0.98, 'Calm',       fontsize=9, color='#666666', ha='right', va='bottom')

ax.legend(title='Group', fontsize=11, title_fontsize=11,
          loc='upper right', framealpha=0.9, edgecolor='#cccccc',
          markerscale=1.5)

plt.tight_layout()
plt.savefig('circumplex_scatter_UNCORRECTED.png', dpi=600, bbox_inches='tight')
plt.show()
print("Saved: circumplex_scatter_UNCORRECTED.png")

In [ ]:
# Cell 8c — Corrected scatter — identical to Cell 8b

import pandas as pd
import matplotlib.pyplot as plt

GROUP_COLORS = {
    'ADHD':         '#1f77b4',
    'Autistic':     '#ff7f0e',
    'Neurotypical': '#2ca02c',
}

frames = []
for group, df in [('ADHD', adhd), ('Autistic', autistic), ('Neurotypical', neurotypical)]:
    tmp = df.copy()
    tmp['Group'] = group
    frames.append(tmp)

combined = pd.concat(frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 10))

for group, color in GROUP_COLORS.items():
    subset = combined[combined['Group'] == group]
    ax.scatter(
        2 * subset['ISOPleasant'] - 1,
        2 * subset['ISOEventful'] - 1,
        color=color, alpha=0.6, s=20, label=group, zorder=3
    )

ax.axhline(0, color='#555555', linewidth=0.8, linestyle='--', zorder=2)
ax.axvline(0, color='#555555', linewidth=0.8, linestyle='--', zorder=2)
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_xlabel('$P_{ISO}$', fontsize=13)
ax.set_ylabel('$E_{ISO}$', fontsize=13)
ax.set_title('Soundscape Perception by Group', fontsize=15)
ax.set_xticks([-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1])
ax.set_yticks([-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1])
ax.minorticks_on()
ax.xaxis.set_minor_locator(plt.MultipleLocator(0.05))
ax.yaxis.set_minor_locator(plt.MultipleLocator(0.05))
ax.grid(True, which='major', color='#9999bb', linewidth=1.0, linestyle=':', alpha=1.0)
ax.grid(True, which='minor', color='#9999bb', linewidth=0.5, linestyle=':', alpha=1.0)
ax.set_axisbelow(True)
ax.text(-0.98,  0.98, 'Chaotic',    fontsize=9, color='#666666', ha='left',  va='top')
ax.text( 0.98,  0.98, 'Vibrant',    fontsize=9, color='#666666', ha='right', va='top')
ax.text(-0.98, -0.98, 'Monotonous', fontsize=9, color='#666666', ha='left',  va='bottom')
ax.text( 0.98, -0.98, 'Calm',       fontsize=9, color='#666666', ha='right', va='bottom')
ax.legend(title='Group', fontsize=11, title_fontsize=11,
          loc='upper right', framealpha=0.9, edgecolor='#cccccc', markerscale=1.5)

plt.tight_layout()
plt.savefig('circumplex_scatter_CORRECTED.png', dpi=600, bbox_inches='tight')
plt.show()
print("Saved: circumplex_scatter_CORRECTED.png")

## One-sweep results

All headline tests in a single cell: group differences (raw vs calibrated, to show invariance), Kruskal-Wallis, pleasantness skewness, and the new response-extremity (radial) tests. Self-contained — only needs `all_data` from the prepare_data cell.

In [ ]:
# ============================================================================
#  ONE-SWEEP RESULTS  —  raw vs calibrated, all key tests in one place
#  Self-contained: only needs `all_data` (Circumplex X/Y + Group) from Cell 5.
# ============================================================================
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu, kruskal, skew

GROUPS = ['ADHD', 'Autistic', 'Neurotypical']

d = all_data.copy()
# RAW centred (old (v-0.5)*2)            CALIBRATED centred (4-point click test)
d['P_raw'] = (d['Circumplex X'] - 0.5) * 2
d['E_raw'] = ((1 - d['Circumplex Y']) - 0.5) * 2
d['P_cal'] = (d['Circumplex X'] - 0.516) / 0.300
d['E_cal'] = (0.507 - d['Circumplex Y']) / 0.307
d['r_raw'] = np.hypot(d['P_raw'], d['E_raw'])
d['r_cal'] = np.hypot(d['P_cal'], d['E_cal'])

def col(g, c): return d.loc[d['Group'] == g, c]
line = "-" * 68

print("="*68); print("  SOUNDSCAPE RESULTS — ONE SWEEP"); print("="*68)
print(f"n per group:  " + "   ".join(f"{g}={ (d['Group']==g).sum() }" for g in GROUPS))
print(f"Calibrated ranges:  P_ISO [{d.P_cal.min():.2f}, {d.P_cal.max():.2f}]   "
      f"E_ISO [{d.E_cal.min():.2f}, {d.E_cal.max():.2f}]")

# ---- A. Group differences vs Neurotypical (invariance check) ----
print("\n" + line)
print("A. GROUP DIFFERENCES vs NEUROTYPICAL  (Mann-Whitney, two-sided)")
print("   raw and calibrated are identical -> result is not a scaling artefact")
print(line)
print(f"{'contrast':<26}{'axis':<12}{'p (raw)':>10}{'p (cal)':>10}")
for axis, raw_c, cal_c in [('Eventfulness','E_raw','E_cal'), ('Pleasantness','P_raw','P_cal')]:
    for g in ['ADHD', 'Autistic']:
        _, p_raw = mannwhitneyu(col(g, raw_c), col('Neurotypical', raw_c), alternative='two-sided')
        _, p_cal = mannwhitneyu(col(g, cal_c), col('Neurotypical', cal_c), alternative='two-sided')
        star = ' *' if p_cal < 0.05 else ''
        print(f"{g+' vs NT':<26}{axis:<12}{p_raw:>10.4f}{p_cal:>10.4f}{star}")

# ---- B. Kruskal-Wallis across all three groups ----
print("\n" + line); print("B. KRUSKAL-WALLIS across 3 groups"); print(line)
for axis, cal_c in [('Eventfulness','E_cal'), ('Pleasantness','P_cal')]:
    H, p = kruskal(*[col(g, cal_c) for g in GROUPS])
    print(f"  {axis:<14} H={H:6.3f}   p={p:.4f}{' *' if p<0.05 else ''}")

# ---- C. Pleasantness skewness per group (invariant under linear scaling) ----
print("\n" + line); print("C. PLEASANTNESS SKEWNESS per group"); print(line)
print(f"{'group':<16}{'skew (raw)':>12}{'skew (cal)':>12}")
for g in GROUPS:
    print(f"{g:<16}{skew(col(g,'P_raw')):>12.3f}{skew(col(g,'P_cal')):>12.3f}")

# ---- D. Response extremity (NEW — only meaningful after calibration) ----
print("\n" + line); print("D. RESPONSE EXTREMITY  r = sqrt(P^2+E^2)  [calibrated]"); print(line)
print(f"{'group':<16}{'median r':>10}{'mean r':>10}{'sd':>8}")
for g in GROUPS:
    rr = col(g, 'r_cal')
    print(f"{g:<16}{rr.median():>10.3f}{rr.mean():>10.3f}{rr.std():>8.3f}")
H, p = kruskal(*[col(g, 'r_cal') for g in GROUPS])
print(f"\n  Kruskal-Wallis (r): H={H:.3f}, p={p:.4f}{' *' if p<0.05 else ''}")
for g in ['ADHD', 'Autistic']:
    U, pu = mannwhitneyu(col(g,'r_cal'), col('Neurotypical','r_cal'), alternative='two-sided')
    print(f"  {g} vs NT:  U={U:.0f}, p={pu:.4f}{' *' if pu<0.05 else ''}")
print("="*68)


In [ ]:
# Cell 23a — Pooled across all soundscapes (group comparison on the full dataset)
from scipy.stats import kruskal, mannwhitneyu
import numpy as np
import pandas as pd

def star(p):
    if pd.isna(p): return ''
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

GROUPS = ['ADHD', 'Autistic', 'Neurotypical']
g = {name: all_data[all_data['Group'] == name] for name in GROUPS}

rows = []
for col in ['ISOEventful', 'ISOPleasant']:
    a, au, nt = (g[n][col].dropna() for n in GROUPS)
    H, p_kw = kruskal(a, au, nt)                              # omnibus, 3 groups
    _, p_an = mannwhitneyu(a, nt, alternative='two-sided')   # ADHD vs NT
    _, p_aa = mannwhitneyu(a, au, alternative='two-sided')   # ADHD vs Autistic
    _, p_un = mannwhitneyu(au, nt, alternative='two-sided')  # Autistic vs NT
    rows.append({
        'Dimension': col,
        'n_ADHD': len(a), 'n_Aut': len(au), 'n_NT': len(nt),
        'med_ADHD': a.median(), 'med_Aut': au.median(), 'med_NT': nt.median(),
        'KW_H': H, 'KW_p': p_kw, 'KW_sig': star(p_kw),
        'ADHDvNT_p': p_an,  'ADHDvNT_sig': star(p_an),
        'ADHDvAut_p': p_aa, 'AutvNT_p': p_un,
    })

pooled = pd.DataFrame(rows).round(3)

print("=" * 80)
print("POOLED — all ten soundscapes combined")
print("=" * 80)
display(pooled[['Dimension','n_ADHD','n_Aut','n_NT',
                'med_ADHD','med_Aut','med_NT',
                'KW_H','KW_p','KW_sig',
                'ADHDvNT_p','ADHDvNT_sig','ADHDvAut_p','AutvNT_p']])

ev = pooled[pooled['Dimension'] == 'ISOEventful'].iloc[0]
direction = "lower" if ev['med_ADHD'] < ev['med_NT'] else "higher"
print(f"\nHeadline: ADHD median Eventful {ev['med_ADHD']} vs NT {ev['med_NT']} "
      f"\u2192 ADHD {direction}; ADHD vs NT p = {ev['ADHDvNT_p']} {ev['ADHDvNT_sig']}")

pooled.to_csv('pooled_stats.csv', index=False)
print("\nSaved: pooled_stats.csv")

In [ ]:
# Cell 23b — Per-Soundscape Statistical Tests (group comparison within each soundscape)
from scipy.stats import kruskal, mannwhitneyu
import numpy as np
import pandas as pd

def bh_fdr(pvals):
    """Benjamini-Hochberg FDR. NaN-safe; returns adjusted p-values."""
    p = np.asarray(pvals, dtype=float)
    out = np.full(p.shape, np.nan)
    idx = np.where(~np.isnan(p))[0]
    if len(idx) == 0:
        return out
    pp = p[idx]; n = len(pp)
    order = np.argsort(pp)
    ranked = pp[order] * n / (np.arange(1, n + 1))
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adj = np.empty(n); adj[order] = np.clip(ranked, 0, 1)
    out[idx] = adj
    return out

def star(p):
    if pd.isna(p): return ''
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

GROUPS = ['ADHD', 'Autistic', 'Neurotypical']
soundscapes = sorted(all_data['Soundscape'].unique())
rows = []

for ss in soundscapes:
    sub = all_data[all_data['Soundscape'] == ss]
    g = {name: sub[sub['Group'] == name] for name in GROUPS}
    for col in ['ISOPleasant', 'ISOEventful']:
        samples = [g[name][col].dropna() for name in GROUPS]
        ns = [len(s) for s in samples]
        H, p_kw = (kruskal(*samples) if all(n >= 2 for n in ns) else (np.nan, np.nan))
        a, nt = samples[0], samples[2]
        U, p_an = (mannwhitneyu(a, nt, alternative='two-sided')
                   if len(a) >= 2 and len(nt) >= 2 else (np.nan, np.nan))
        rows.append({
            'Soundscape': ss, 'Dimension': col,
            'n_ADHD': ns[0], 'n_Aut': ns[1], 'n_NT': ns[2],
            'med_ADHD': samples[0].median(),
            'med_Aut':  samples[1].median(),
            'med_NT':   samples[2].median(),
            'KW_H': H, 'KW_p': p_kw,
            'ADHDvNT_U': U, 'ADHDvNT_p': p_an,
        })

res = pd.DataFrame(rows)

res['KW_p_FDR'] = np.nan
res['ADHDvNT_p_FDR'] = np.nan
for col in ['ISOPleasant', 'ISOEventful']:
    m = res['Dimension'] == col
    res.loc[m, 'KW_p_FDR'] = bh_fdr(res.loc[m, 'KW_p'].values)
    res.loc[m, 'ADHDvNT_p_FDR'] = bh_fdr(res.loc[m, 'ADHDvNT_p'].values)

res['KW_sig'] = res['KW_p'].apply(star)
res['ADHDvNT_sig'] = res['ADHDvNT_p'].apply(star)
res = res.round(3)

for col in ['ISOEventful', 'ISOPleasant']:
    print(f"\n{'='*80}\nPER-SOUNDSCAPE — {col}\n{'='*80}")
    view = res[res['Dimension'] == col][[
        'Soundscape', 'n_ADHD', 'n_Aut', 'n_NT',
        'med_ADHD', 'med_Aut', 'med_NT',
        'KW_H', 'KW_p', 'KW_sig', 'KW_p_FDR',
        'ADHDvNT_p', 'ADHDvNT_sig', 'ADHDvNT_p_FDR',
    ]].reset_index(drop=True)
    display(view)

print("\n" + "="*80)
print("PLAIN-ENGLISH SUMMARY — ADHD vs NT, Eventful")
print("="*80)
ev = res[res['Dimension'] == 'ISOEventful']
hits = ev[ev['ADHDvNT_p'] < 0.05]
if len(hits) == 0:
    print("No single soundscape reaches p < .05 on its own.")
    print("The pooled effect (p = 0.043) is spread across soundscapes rather than")
    print("driven by one or two. That even spread is itself a reportable finding.")
else:
    for _, r in hits.iterrows():
        direction = "lower" if r['med_ADHD'] < r['med_NT'] else "higher"
        survives = "survives FDR" if r['ADHDvNT_p_FDR'] < 0.05 else "does NOT survive FDR"
        print(f"  {r['Soundscape']}: ADHD {direction} than NT "
              f"(raw p = {r['ADHDvNT_p']:.3f}, FDR = {r['ADHDvNT_p_FDR']:.3f}, {survives})")

res.to_csv('per_soundscape_stats.csv', index=False)
print("\nSaved: per_soundscape_stats.csv")

In [ ]:
# Cell 23c — Per-Soundscape Plots (600 dpi)
import matplotlib.pyplot as plt
import numpy as np

DPI = 600
GROUP_COLOURS = {'ADHD': '#E69F00', 'Autistic': '#56B4E9', 'Neurotypical': '#009E73'}

ev = res[res['Dimension'] == 'ISOEventful'].set_index('Soundscape')

# --- Plot 1: median Eventful per soundscape, by group ---
order = ev['med_NT'].sort_values().index
x = np.arange(len(order))
fig, ax = plt.subplots(figsize=(10, 6))
for grp, col in [('ADHD', 'med_ADHD'), ('Autistic', 'med_Aut'), ('Neurotypical', 'med_NT')]:
    ax.plot(x, ev.loc[order, col], 'o-', color=GROUP_COLOURS[grp], label=grp,
            lw=1.2, ms=7, alpha=0.9)
ax.axhline(0, color='grey', lw=0.8, ls='--', alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=45, ha='right')
ax.set_ylabel('Median ISOEventful (calibrated)')
ax.set_title('Median Eventfulness per Soundscape, by Group')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('per_soundscape_eventful_medians.png', dpi=DPI, bbox_inches='tight')
plt.show()

# --- Plot 2: ADHD - NT Eventful gap per soundscape (forest) ---
ev2 = ev.copy()
ev2['gap'] = ev2['med_ADHD'] - ev2['med_NT']
ev2 = ev2.sort_values('gap')
y = np.arange(len(ev2))
fig, ax = plt.subplots(figsize=(8, 6))
colours = ['#E69F00' if g < 0 else '#999999' for g in ev2['gap']]
ax.barh(y, ev2['gap'], color=colours, alpha=0.85)
for yi in range(len(ev2)):
    sig = ev2['ADHDvNT_sig'].iloc[yi]; g = ev2['gap'].iloc[yi]
    if sig and sig != 'ns':
        ax.text(g, yi, f'  {sig}', va='center',
                ha='left' if g >= 0 else 'right', fontsize=9)
ax.axvline(0, color='black', lw=0.8)
ax.set_yticks(y); ax.set_yticklabels(ev2.index)
ax.set_xlabel('Median Eventful gap  (ADHD − NT)')
ax.set_title('ADHD vs NT Eventfulness gap, per soundscape\n(negative = ADHD less eventful)')
plt.tight_layout()
plt.savefig('per_soundscape_eventful_gap.png', dpi=DPI, bbox_inches='tight')
plt.show()

print("Saved at 600 dpi:")
print("  per_soundscape_eventful_medians.png")
print("  per_soundscape_eventful_gap.png")

In [ ]:
# Cell 23d — Interactive Circumplex (click a group to isolate it)
import plotly.graph_objects as go

d = all_data.copy()
d['P'] = (d['ISOPleasant'] - 0.5) * 2
d['E'] = (d['ISOEventful'] - 0.5) * 2

COLOURS = {'ADHD': '#1f77b4', 'Autistic': '#ff7f0e', 'Neurotypical': '#2ca02c'}
GROUPS = ['ADHD', 'Autistic', 'Neurotypical']

fig = go.Figure()
for grp in GROUPS:
    sub = d[d['Group'] == grp]
    fig.add_trace(go.Scatter(
        x=sub['P'], y=sub['E'], mode='markers', name=grp,
        marker=dict(color=COLOURS[grp], size=6, opacity=0.7, line=dict(width=0)),
        hovertext=sub['Soundscape'], hoverinfo='text+name',
    ))

line = dict(color='black', width=1, dash='dash')
fig.add_shape(type='line', x0=0, x1=0, y0=-1, y1=1, line=line)
fig.add_shape(type='line', x0=-1, x1=1, y0=0, y1=0, line=line)
diag = dict(color='lightgrey', width=1, dash='dot')
fig.add_shape(type='line', x0=-1, x1=1, y0=-1, y1=1, line=diag)
fig.add_shape(type='line', x0=-1, x1=1, y0=1, y1=-1, line=diag)

for x, y, t, ax, ay in [(-1, 1, 'Chaotic', 'left', 'top'),
                        (1, 1, 'Vibrant', 'right', 'top'),
                        (-1, -1, 'Monotonous', 'left', 'bottom'),
                        (1, -1, 'Calm', 'right', 'bottom')]:
    fig.add_annotation(x=x, y=y, text=t, showarrow=False,
                       xanchor=ax, yanchor=ay, font=dict(size=11, color='#666'))

def vis(g): return [g == 'All' or g == grp for grp in GROUPS]
buttons = [dict(label=lbl, method='update', args=[{'visible': vis(lbl)}])
           for lbl in ['All'] + GROUPS]

grid = dict(showgrid=True, gridcolor='#cccccc', dtick=0.5, zeroline=False,
            minor=dict(dtick=0.25, showgrid=True, gridcolor='#eeeeee'))

fig.update_layout(
    title=dict(text='Soundscape Perception by Group', x=0.5, y=0.97, yanchor='top'),
    xaxis=dict(title='P_ISO', range=[-1, 1], constrain='domain', **grid),
    yaxis=dict(title='E_ISO', range=[-1, 1], scaleanchor='x', scaleratio=1,
               constrain='domain', **grid),
    width=900, height=820, template='plotly_white',
    margin=dict(l=70, r=170, t=120, b=70),
    legend=dict(title='Group  (click to toggle)', x=1.02, y=1, xanchor='left'),
    updatemenus=[dict(type='buttons', direction='right', x=0, y=1.15,
                      xanchor='left', yanchor='top', buttons=buttons, showactive=True)],
)

fig.show()
fig.write_html('circumplex_interactive.html', include_plotlyjs='inline', full_html=True)
print("Saved: circumplex_interactive.html")


In [ ]:
# ============================================================================
#  EXPORT EVERYTHING  —  all figures @ 600 dpi + all tables to /exports
#  Run this last (after Run All). Self-contained: rebuilds from `all_data`.
# ============================================================================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import mannwhitneyu, kruskal, skew

OUT = "exports"; os.makedirs(OUT, exist_ok=True)
DPI = 600
GROUPS = ['ADHD', 'Autistic', 'Neurotypical']

# ---- rebuild coordinates (raw + calibrated) from all_data ----
d = all_data.copy()
d['P_raw'] = (d['Circumplex X'] - 0.5) * 2
d['E_raw'] = ((1 - d['Circumplex Y']) - 0.5) * 2
d['P_cal'] = (d['Circumplex X'] - 0.516) / 0.300
d['E_cal'] = (0.507 - d['Circumplex Y']) / 0.307
d['r_cal'] = np.hypot(d['P_cal'], d['E_cal'])
def col(g, c): return d.loc[d['Group'] == g, c]

def circumplex_axes(ax, title):
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_aspect('equal')
    ax.axhline(0, color='#555', lw=.8, ls='--'); ax.axvline(0, color='#555', lw=.8, ls='--')
    ax.plot([-1, 1], [-1, 1], color='grey', lw=1, ls='--', alpha=.5)
    ax.plot([-1, 1], [1, -1], color='grey', lw=1, ls='--', alpha=.5)
    lk = dict(fontsize=10, color='#555', bbox=dict(facecolor='white', edgecolor='none', alpha=.7, pad=2))
    ax.text( .97,  .97, 'Vibrant',    ha='right', va='top',    **lk)
    ax.text(-.97,  .97, 'Chaotic',    ha='left',  va='top',    **lk)
    ax.text(-.97, -.97, 'Monotonous', ha='left',  va='bottom', **lk)
    ax.text( .97, -.97, 'Calm',       ha='right', va='bottom', **lk)
    ax.set_xlabel('$P_{ISO}$'); ax.set_ylabel('$E_{ISO}$'); ax.set_title(title, weight='bold')

# ---- FIGURE 1: calibrated group scatter (soundscapy house style) ----
# build a FRESH frame (avoids duplicate ISOPleasant/ISOEventful columns in all_data)
plot_df = pd.DataFrame({'ISOPleasant': d['P_cal'].to_numpy(),
                        'ISOEventful': d['E_cal'].to_numpy(),
                        'Group':       d['Group'].to_numpy()})
fig, ax = plt.subplots(figsize=(8, 8))
sspy.plotting.scatter(data=plot_df, x='ISOPleasant', y='ISOEventful', hue='Group',
                      palette=GROUP_COLOURS, diagonal_lines=False,
                      title='Soundscape Perception by Group (calibrated)',
                      xlim=(-1, 1), ylim=(-1, 1), s=50, alpha=0.6, ax=ax)
circumplex_axes(ax, 'Soundscape Perception by Group (calibrated)')
plt.tight_layout(); plt.savefig(f"{OUT}/fig1_scatter_calibrated.png", dpi=DPI, bbox_inches='tight'); plt.close()

# ---- FIGURE 2: per-group density panels ----
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for axn, g in zip(axes, GROUPS):
    sub = d[d['Group'] == g]
    sns.kdeplot(x=sub['P_cal'], y=sub['E_cal'], ax=axn, fill=True,
                color=GROUP_COLOURS[g], thresh=.05, levels=8, alpha=.8)
    circumplex_axes(axn, f"{g} (n={ (d['Group']==g).sum() })")
plt.tight_layout(); plt.savefig(f"{OUT}/fig2_density_by_group.png", dpi=DPI, bbox_inches='tight'); plt.close()

# ---- FIGURE 3: response extremity (radial distribution) ----
fig, ax = plt.subplots(figsize=(9, 5.5))
for g in GROUPS:
    sns.kdeplot(col(g, 'r_cal'), ax=ax, color=GROUP_COLOURS[g], fill=True,
                alpha=.18, lw=2, label=g, clip=(0, 1.45), bw_adjust=.9)
ax.axvline(1.0, color='grey', ls='--', lw=1, alpha=.6)
ax.set_xlabel(r'Radial distance  $r=\sqrt{P_{ISO}^2+E_{ISO}^2}$'); ax.set_ylabel('Density')
ax.set_title('Response extremity by listener group'); ax.legend(title='Group', frameon=False)
plt.tight_layout(); plt.savefig(f"{OUT}/fig3_radial_extremity.png", dpi=DPI, bbox_inches='tight'); plt.close()

# ---- TABLES ----
rows = []
for axis, rc, cc in [('Eventfulness', 'E_raw', 'E_cal'), ('Pleasantness', 'P_raw', 'P_cal')]:
    for g in ['ADHD', 'Autistic']:
        _, praw = mannwhitneyu(col(g, rc), col('Neurotypical', rc), alternative='two-sided')
        _, pcal = mannwhitneyu(col(g, cc), col('Neurotypical', cc), alternative='two-sided')
        rows.append({'contrast': f'{g} vs NT', 'axis': axis,
                     'p_raw': round(praw, 4), 'p_calibrated': round(pcal, 4)})
t1 = pd.DataFrame(rows)

t2 = (d.groupby('Group')
        .apply(lambda x: pd.Series({
            'E_median': x['E_cal'].median(), 'P_median': x['P_cal'].median(),
            'P_skew': skew(x['P_cal']), 'r_median': x['r_cal'].median(),
            'r_mean': x['r_cal'].mean(), 'r_sd': x['r_cal'].std()}))
        .round(3).reindex(GROUPS))

rows = []
for axis, c in [('Eventfulness', 'E_cal'), ('Pleasantness', 'P_cal'), ('Radius r', 'r_cal')]:
    H, p = kruskal(*[col(g, c) for g in GROUPS])
    rows.append({'measure': axis, 'test': 'Kruskal-Wallis', 'stat': round(H, 3), 'p': round(p, 4)})
for g in ['ADHD', 'Autistic']:
    U, p = mannwhitneyu(col(g, 'r_cal'), col('Neurotypical', 'r_cal'), alternative='two-sided')
    rows.append({'measure': f'r: {g} vs NT', 'test': 'Mann-Whitney', 'stat': round(U, 1), 'p': round(p, 4)})
t3 = pd.DataFrame(rows)

for name, t in [('table1_contrasts_raw_vs_cal', t1),
                ('table2_descriptives_by_group', t2),
                ('table3_omnibus_and_radial', t3)]:
    t.to_csv(f"{OUT}/{name}.csv")
    print(f"\n=== {name} ===\n", t.to_string())

print(f"\nAll figures (600 dpi) + tables saved to ./{OUT}/")

In [ ]:
# Cell 20 — Compute SPI Tables (per soundscape)

def compute_spi_table(df1, df2, g1, g2):
    soundscapes = sorted(set(df1['Soundscape'].unique()) & set(df2['Soundscape'].unique()))
    results = []
    for ss in soundscapes:
        s1 = df1[df1['Soundscape'] == ss]
        s2 = df2[df2['Soundscape'] == ss]
        x1, y1 = s1['ISOPleasant'].values, s1['ISOEventful'].values
        x2, y2 = s2['ISOPleasant'].values, s2['ISOEventful'].values
        if len(x1) >= 3 and len(x2) >= 3:
            D, p = bks_distance_2d(x1, y1, x2, y2)
            results.append({
                'Soundscape': ss,
                f'n_{g1}': len(x1),
                f'n_{g2}': len(x2),
                'D_BKS': round(D, 4),
                'p_value': round(p, 4),
                'SPI': round(compute_spi(D), 2)
            })
    return pd.DataFrame(results)

spi_adhd_nt  = compute_spi_table(adhd, neurotypical, 'ADHD', 'NT')
spi_aut_nt   = compute_spi_table(autistic, neurotypical, 'Autistic', 'NT')
spi_adhd_aut = compute_spi_table(adhd, autistic, 'ADHD', 'Autistic')

print("SPI: ADHD vs Neurotypical")
display(spi_adhd_nt)
print("\nSPI: Autistic vs Neurotypical")
display(spi_aut_nt)
print("\nSPI: ADHD vs Autistic")
display(spi_adhd_aut)